# Legacy HOG Model Evaluation

This notebook evaluates the **HOG (Histogram of Oriented Gradients)** face recognition model
using the `face_recognition` library with the same metrics and methodology as the InsightFace
baseline evaluation notebook.

**Model:** `face_recognition_hog` — HOG face detector + `face_recognition` embeddings + euclidean distance matching

**Metrics Calculated:**
- Subset Accuracy, F-beta (macro/micro), Precision/Recall (macro/micro)
- Precision@Recall=0.95 (macro/micro), Recall@Precision=0.95 (macro/micro)
- Per-class: Precision, Recall, F-beta, ROC-AUC, P@R, R@P, confusion values
- PR curves, ROC curves, heatmaps

In [1]:
import datetime
import json
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Add project root to path and change working directory so relative image paths work
sys.path.insert(0, str(Path("/app")))
os.chdir("/app")

from src.classification import get_classifier, load_celebrities_from_json

BASE_DIR = Path("/app")
TRAINSET_PATH = BASE_DIR / "testsets/four-people-trainset.json"
REFERENCES_PATH = BASE_DIR / "data/references.json"
OUTPUT_ROOT = BASE_DIR / "image_outputs"

# Model config
MODEL_KEY = "face_recognition_hog"
MODEL_DISPLAY = "HOG (face_recognition)"
MODEL_COLOR = '#e74c3c'

# Detection config (HOG-specific)
DETECTION_MODEL = "hog"
EMBEDDING_MODEL = "face_recognition"
MATCHING_METHOD = "cosine_similarity"  # Same as InsightFace notebook

# Create experiment output directory
EXPERIMENT_DIR = OUTPUT_ROOT / f"eval_hog_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation parameters — identical to the InsightFace evaluation notebook
FACE_IDENTIFICATION_THRESHOLD = 0.30   # Cosine similarity threshold (same as InsightFace)
F1_BETA = 0.4
FIXED_RECALL_LEVEL = 0.95
FIXED_PRECISION_LEVEL = 0.95

print(f"\u2713 Setup complete")
print(f"\nModel: {MODEL_DISPLAY}")
print(f"Matching: {MATCHING_METHOD} (threshold={FACE_IDENTIFICATION_THRESHOLD})")
print(f"Output directory: {EXPERIMENT_DIR}")
print(f"Working directory: {os.getcwd()}")
print(f"\nEvaluation Configuration:")
print(f"  Face identification threshold: {FACE_IDENTIFICATION_THRESHOLD}")
print(f"  Similarity method: {MATCHING_METHOD}")
print(f"  F-beta parameter: {F1_BETA}")
print(f"  Fixed recall level: {FIXED_RECALL_LEVEL}")
print(f"  Fixed precision level: {FIXED_PRECISION_LEVEL}")

/opt/venv/lib/python3.12/site-packages/face_recognition_models/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Setup complete

Model: HOG (face_recognition)
Matching: cosine_similarity (threshold=0.3)
Output directory: /app/image_outputs/eval_hog_20260310_205046
Working directory: /app

Evaluation Configuration:
  Face identification threshold: 0.3
  Similarity method: cosine_similarity
  F-beta parameter: 0.4
  Fixed recall level: 0.95
  Fixed precision level: 0.95


## Load Training Set and References

In [2]:
# Load training set
with open(TRAINSET_PATH, "r", encoding="utf-8") as f:
    train_items = json.load(f)

print(f"\u2713 Loaded {len(train_items)} training images")

# Analyze training set distribution
label_counts = Counter()
for item in train_items:
    for label in item.get("labels", []):
        label_counts[label] += 1

print(f"\nTraining set label distribution:")
for label, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count} images")

# Load celebrity reference data
celebrity_data = load_celebrities_from_json(str(REFERENCES_PATH))
print(f"\n\u2713 Loaded {len(celebrity_data)} celebrity reference(s)")
for cd in celebrity_data:
    print(f"  - {cd['name']}: {cd['reference_image_path']}")

# Identity list
with open(REFERENCES_PATH, "r", encoding="utf-8") as f:
    base_references = json.load(f)
all_identities = [ref["name"] for ref in base_references]
print(f"\nIdentities: {all_identities}")

2026-03-10 20:50:46,512 - src.logging_utils - INFO - Loading celebrity data from /app/data/references.json
2026-03-10 20:50:46,517 - src.logging_utils - INFO - Successfully loaded 7 celebrity reference(s) from /app/data/references.json


✓ Loaded 1536 training images

Training set label distribution:
  Lionel Messi: 369 images
  Donald Trump: 351 images
  Giorgia Meloni: 350 images
  Hugh Jackman: 304 images
  None: 228 images

✓ Loaded 7 celebrity reference(s)
  - Hugh Jackman: Images/references/HughJackman.jpg
  - Donald Trump: Images/references/DonaldTrump.jpg
  - Giorgia Meloni: Images/references/GiorgiaMeloni.jpg
  - Lionel Messi: Images/references/LionnelMessi.png
  - Javier Bardem: Images/references/JavierBardem.jpg
  - Jeffrey Dean Morgan: Images/references/JeffreyDeanMorgan.jpg
  - Samantha Cristoforetti: Images/references/SamanthaCristoforetti.jpg

Identities: ['Hugh Jackman', 'Donald Trump', 'Giorgia Meloni', 'Lionel Messi', 'Javier Bardem', 'Jeffrey Dean Morgan', 'Samantha Cristoforetti']


## Define Evaluation Functions

In [3]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, fbeta_score,
    precision_recall_curve, roc_curve, auc as sk_auc,
)


def cosine_similarity(emb1, emb2):
    """Cosine similarity between two embeddings (same as InsightFace notebook)."""
    norm1 = emb1 / (np.linalg.norm(emb1) + 1e-8)
    norm2 = emb2 / (np.linalg.norm(emb2) + 1e-8)
    return float(np.dot(norm1, norm2))


def evaluate_model_with_scores(classifier, train_items, all_identities, threshold):
    """Run classifier on all images and return predictions + per-identity cosine similarity scores."""
    all_predictions = []
    all_ground_truth = []
    all_prediction_scores = []

    total_faces = 0
    identified_faces = 0
    unknown_faces = 0
    no_face_count = 0

    ref_embeddings = np.array(classifier.reference_embeddings)
    ref_names = classifier.reference_names

    for item in tqdm(train_items, desc=f"Evaluating ({classifier.name})"):
        img_path_str = item.get("path", "")
        ground_truth = set(item.get("labels", []))

        predicted_set = set()
        image_scores = {identity: [] for identity in all_identities}

        img = cv2.imread(img_path_str)
        if img is None:
            no_face_count += 1
            predicted_set.add("None")
            all_ground_truth.append(list(ground_truth))
            all_predictions.append(list(predicted_set))
            all_prediction_scores.append({identity: 0.0 for identity in all_identities})
            continue

        face_bboxes = classifier.face_detector.detect_faces(img)
        if not face_bboxes:
            no_face_count += 1
            predicted_set.add("None")
            all_ground_truth.append(list(ground_truth))
            all_predictions.append(list(predicted_set))
            all_prediction_scores.append({identity: 0.0 for identity in all_identities})
            continue

        if (classifier.use_alignment
            and hasattr(classifier, 'embedder')
            and hasattr(classifier.embedder, 'extract_embedding_aligned')
            and classifier.face_detector.model in classifier.face_detector.INSIGHTFACE_MODELS):
            face_infos = classifier.face_detector.detect_faces_with_landmarks(img)
            embeddings = [classifier.embedder.extract_embedding_aligned(img, fi) for fi in face_infos]
        else:
            embeddings = classifier.embedder.extract_embeddings_batch(img, face_bboxes)

        for embedding in embeddings:
            if embedding is None:
                continue
            total_faces += 1

            identity_scores = {}
            # Cosine similarity — same method as InsightFace notebook
            for ref_emb, ref_name in zip(ref_embeddings, ref_names):
                score = cosine_similarity(embedding, ref_emb)
                if ref_name not in identity_scores or score > identity_scores[ref_name]:
                    identity_scores[ref_name] = score

            for identity in all_identities:
                s = identity_scores.get(identity, 0.0)
                image_scores[identity].append(s)

            best_match = max(identity_scores, key=identity_scores.get) if identity_scores else None
            best_score = identity_scores.get(best_match, 0.0) if best_match else 0.0
            if best_score >= threshold:
                predicted_set.add(best_match)
                identified_faces += 1
            else:
                unknown_faces += 1

        if not predicted_set:
            predicted_set.add("None")

        all_ground_truth.append(list(ground_truth))
        all_predictions.append(list(predicted_set))

        max_scores = {identity: max(scores) if scores else 0.0
                      for identity, scores in image_scores.items()}
        all_prediction_scores.append(max_scores)

    return {
        "all_predictions": all_predictions,
        "all_ground_truth": all_ground_truth,
        "all_prediction_scores": all_prediction_scores,
        "total_faces": total_faces,
        "identified_faces": identified_faces,
        "unknown_faces": unknown_faces,
        "no_face_count": no_face_count,
    }


def compute_full_metrics(eval_result, all_identities, f_beta, fixed_recall, fixed_precision):
    """Compute comprehensive metrics identical to the InsightFace evaluation notebook."""
    all_ground_truth = eval_result["all_ground_truth"]
    all_predictions = eval_result["all_predictions"]
    all_prediction_scores = eval_result["all_prediction_scores"]

    mlb = MultiLabelBinarizer()
    y_true = mlb.fit_transform(all_ground_truth)
    y_pred = mlb.transform(all_predictions)

    subset_accuracy = float(accuracy_score(y_true, y_pred))
    f_beta_macro = float(fbeta_score(y_true, y_pred, beta=f_beta, average='macro', zero_division=0))
    f_beta_micro = float(fbeta_score(y_true, y_pred, beta=f_beta, average='micro', zero_division=0))
    precision_macro = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
    precision_micro = float(precision_score(y_true, y_pred, average='micro', zero_division=0))
    recall_macro = float(recall_score(y_true, y_pred, average='macro', zero_division=0))
    recall_micro = float(recall_score(y_true, y_pred, average='micro', zero_division=0))

    all_possible_labels = (set(c for sub in all_ground_truth for c in sub)
                           | set(c for sub in all_predictions for c in sub))
    mlb_full = MultiLabelBinarizer()
    mlb_full.fit([list(all_possible_labels)])
    y_true_full = mlb_full.transform(all_ground_truth)
    y_pred_full = mlb_full.transform(all_predictions)

    beta2 = f_beta ** 2
    per_class_metrics = {}
    for i, class_name in enumerate(mlb_full.classes_):
        tp = int(np.sum((y_pred_full[:, i] == 1) & (y_true_full[:, i] == 1)))
        fp = int(np.sum((y_pred_full[:, i] == 1) & (y_true_full[:, i] == 0)))
        fn = int(np.sum((y_pred_full[:, i] == 0) & (y_true_full[:, i] == 1)))
        tn = int(np.sum((y_pred_full[:, i] == 0) & (y_true_full[:, i] == 0)))

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
        fb = ((1 + beta2) * prec * rec / (beta2 * prec + rec)) if (prec + rec) > 0 else 0

        y_true_binary = y_true_full[:, i]
        if class_name == 'None':
            y_scores = np.array([1 - max(s.values()) if s else 1.0 for s in all_prediction_scores])
        else:
            y_scores = np.array([s.get(class_name, 0.0) for s in all_prediction_scores])

        prec_curve, rec_curve, _ = precision_recall_curve(y_true_binary, y_scores)
        p_at_fixed_r = float(np.interp(fixed_recall, rec_curve[::-1], prec_curve[::-1], left=0.0, right=0.0))

        valid_recalls = rec_curve[prec_curve >= fixed_precision]
        r_at_fixed_p = float(valid_recalls.max()) if len(valid_recalls) > 0 else 0.0

        fpr, tpr, _ = roc_curve(y_true_binary, y_scores)
        roc_auc = sk_auc(fpr, tpr)

        per_class_metrics[class_name] = {
            "precision": float(prec), "recall": float(rec), "f_beta": float(fb),
            "accuracy": float(acc), "support": tp + fn,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision_at_fixed_recall": p_at_fixed_r,
            "recall_at_fixed_precision": r_at_fixed_p,
            "roc_auc": float(roc_auc),
        }

    # Macro-averaged accuracy and ROC AUC from per-class values
    accuracy_macro = float(np.mean([per_class_metrics[c]["accuracy"] for c in per_class_metrics]))
    roc_auc_macro = float(np.mean([per_class_metrics[c]["roc_auc"] for c in per_class_metrics]))

    p_at_r_macro = np.mean([per_class_metrics[c]["precision_at_fixed_recall"] for c in per_class_metrics])
    r_at_p_macro = np.mean([per_class_metrics[c]["recall_at_fixed_precision"] for c in per_class_metrics])

    y_true_flat = y_true.ravel()
    y_scores_flat = []
    for sample_idx in range(len(all_prediction_scores)):
        for class_idx, class_name in enumerate(mlb.classes_):
            if class_name == 'None':
                score = 1 - max(all_prediction_scores[sample_idx].values()) if all_prediction_scores[sample_idx] else 1.0
            else:
                score = all_prediction_scores[sample_idx].get(class_name, 0.0)
            y_scores_flat.append(score)
    y_scores_flat = np.array(y_scores_flat)

    prec_micro_curve, rec_micro_curve, _ = precision_recall_curve(y_true_flat, y_scores_flat)
    p_at_r_micro = float(np.interp(fixed_recall, rec_micro_curve[::-1], prec_micro_curve[::-1], left=0.0, right=0.0))
    valid_micro = rec_micro_curve[prec_micro_curve >= fixed_precision]
    r_at_p_micro = float(valid_micro.max()) if len(valid_micro) > 0 else 0.0

    total_faces = eval_result["total_faces"]
    identified = eval_result["identified_faces"]

    result = {
        "total_faces": total_faces,
        "identified_faces": identified,
        "unknown_faces": eval_result["unknown_faces"],
        "identification_rate": float(identified / max(total_faces, 1)),
        "subset_accuracy": subset_accuracy,
        "accuracy_macro": accuracy_macro,
        "f_beta_macro": f_beta_macro,
        "f_beta_micro": f_beta_micro,
        "precision_macro": precision_macro,
        "precision_micro": precision_micro,
        "recall_macro": recall_macro,
        "recall_micro": recall_micro,
        "roc_auc_macro": roc_auc_macro,
        "precision_at_fixed_recall_macro": float(p_at_r_macro),
        "precision_at_fixed_recall_micro": float(p_at_r_micro),
        "recall_at_fixed_precision_macro": float(r_at_p_macro),
        "recall_at_fixed_precision_micro": float(r_at_p_micro),
    }

    detailed = {
        "y_true": y_true,
        "y_pred": y_pred,
        "scores": all_prediction_scores,
        "ground_truth": all_ground_truth,
        "predictions": all_predictions,
        "class_names": mlb.classes_.tolist(),
        "per_class_metrics": per_class_metrics,
    }

    return result, detailed

print("\u2713 Evaluation functions defined (using cosine similarity)")

✓ Evaluation functions defined (using cosine similarity)


## Run Evaluation — HOG Model

In [4]:
print(f"{'=' * 80}")
print(f"EVALUATING: {MODEL_DISPLAY} (cosine similarity, threshold={FACE_IDENTIFICATION_THRESHOLD})")
print(f"{'=' * 80}")

classifier = get_classifier(
    "unified", celebrity_data,
    detection_model=DETECTION_MODEL,
    embedding_model=EMBEDDING_MODEL,
    matching_method=MATCHING_METHOD,
)
print(f"\u2713 Classifier initialised: {classifier.name}")
print(f"  Detection: {DETECTION_MODEL} | Embedding: {EMBEDDING_MODEL} | Matching: {MATCHING_METHOD}")
print(f"  Reference embeddings: {len(classifier.reference_embeddings)}")

assert len(classifier.reference_embeddings) > 0, "No reference embeddings loaded — check reference image paths"

eval_result = evaluate_model_with_scores(
    classifier, train_items, all_identities,
    threshold=FACE_IDENTIFICATION_THRESHOLD,
)

print(f"\n  Total faces detected: {eval_result['total_faces']}")
print(f"  Identified: {eval_result['identified_faces']}")
print(f"  Unknown: {eval_result['unknown_faces']}")
print(f"  No face: {eval_result['no_face_count']}")

result, detailed = compute_full_metrics(
    eval_result, all_identities,
    f_beta=F1_BETA,
    fixed_recall=FIXED_RECALL_LEVEL,
    fixed_precision=FIXED_PRECISION_LEVEL,
)
result["model"] = MODEL_KEY
result["display_name"] = MODEL_DISPLAY

print(f"\n  \u2500\u2500 Results \u2500\u2500")
print(f"  Subset Accuracy:         {result['subset_accuracy']:.4f}")
print(f"  Accuracy (macro):        {result['accuracy_macro']:.4f}")
print(f"  F-beta (macro, \u03b2={F1_BETA}):  {result['f_beta_macro']:.4f}")
print(f"  F-beta (micro, \u03b2={F1_BETA}):  {result['f_beta_micro']:.4f}")
print(f"  Precision (macro):       {result['precision_macro']:.4f}")
print(f"  Precision (micro):       {result['precision_micro']:.4f}")
print(f"  Recall (macro):          {result['recall_macro']:.4f}")
print(f"  Recall (micro):          {result['recall_micro']:.4f}")
print(f"  ROC AUC (macro):         {result['roc_auc_macro']:.4f}")
print(f"  P@R={FIXED_RECALL_LEVEL} (macro):      {result['precision_at_fixed_recall_macro']:.4f}")
print(f"  P@R={FIXED_RECALL_LEVEL} (micro):      {result['precision_at_fixed_recall_micro']:.4f}")
print(f"  R@P={FIXED_PRECISION_LEVEL} (macro):      {result['recall_at_fixed_precision_macro']:.4f}")
print(f"  R@P={FIXED_PRECISION_LEVEL} (micro):      {result['recall_at_fixed_precision_micro']:.4f}")

del classifier

2026-03-10 20:50:46,792 - src.logging_utils - INFO - Getting classifier: unified
2026-03-10 20:50:46,794 - src.logging_utils - INFO - Initializing UnifiedClassifier: unified_hog_face_recognition
  Detection: hog
  Embedding: face_recognition
  Matching: cosine_similarity
  Threshold: 0.6
2026-03-10 20:50:46,795 - src.logging_utils - INFO - FaceDetector initialized with model: hog, det_size: (640, 640), multi-pass: False


EVALUATING: HOG (face_recognition) (cosine similarity, threshold=0.3)


2026-03-10 20:50:52,101 - src.logging_utils - INFO - Successfully initialized unified_hog_face_recognition with 7 reference embeddings


✓ Classifier initialised: unified_hog_face_recognition
  Detection: hog | Embedding: face_recognition | Matching: cosine_similarity
  Reference embeddings: 7


Evaluating (unified_hog_face_recognition): 100%|██████████| 1536/1536 [11:27<00:00,  2.23it/s]


  Total faces detected: 2238
  Identified: 2238
  Unknown: 0
  No face: 415

  ── Results ──
  Subset Accuracy:         0.5065
  Accuracy (macro):        0.8529
  F-beta (macro, β=0.4):  0.5831
  F-beta (micro, β=0.4):  0.5715
  Precision (macro):       0.5805
  Precision (micro):       0.5625
  Recall (macro):          0.6322
  Recall (micro):          0.6348
  ROC AUC (macro):         nan
  P@R=0.95 (macro):      0.1698
  P@R=0.95 (micro):      0.2097
  R@P=0.95 (macro):      0.2984
  R@P=0.95 (micro):      0.0000



/opt/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['Javier Bardem', 'Jeffrey Dean Morgan', 'Samantha Cristoforetti'] will be ignored
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1303: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1303: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/metrics/_ran

## Per-Class Performance

In [5]:
pcm = detailed["per_class_metrics"]

print(f"{'=' * 110}")
print(f"PER-CLASS METRICS — {MODEL_DISPLAY}")
print(f"{'=' * 110}")
print(f"  {'Class':<25} {'Acc':>8} {'Prec':>8} {'Recall':>8} {'F-beta':>8} {'ROC-AUC':>8} {'R@P=.95':>8} {'TP':>6} {'FP':>6} {'FN':>6} {'TN':>6}")
print(f"  {'\u2500' * 106}")
for cn in sorted(pcm.keys()):
    m = pcm[cn]
    print(f"  {cn:<25} {m['accuracy']:>8.4f} {m['precision']:>8.4f} {m['recall']:>8.4f} {m['f_beta']:>8.4f} "
          f"{m['roc_auc']:>8.4f} "
          f"{m['recall_at_fixed_precision']:>8.4f} {m['tp']:>6} {m['fp']:>6} {m['fn']:>6} {m['tn']:>6}")
print(f"{'=' * 110}")

PER-CLASS METRICS — HOG (face_recognition)
  Class                          Acc     Prec   Recall   F-beta  ROC-AUC  R@P=.95     TP     FP     FN     TN
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────
  Donald Trump                0.7865   0.5291   0.5954   0.5374   0.6600   0.4245    209    186    142    999
  Giorgia Meloni              0.8275   0.6225   0.6171   0.6217   0.7450   0.5886    216    131    134   1055
  Hugh Jackman                0.8874   0.6804   0.8125   0.6960   0.8730   0.7993    247    116     57   1116
  Javier Bardem               0.9206   0.0000   0.0000   0.0000      nan   0.0000      0    122      0   1414
  Jeffrey Dean Morgan         0.9297   0.0000   0.0000   0.0000      nan   0.0000      0    108      0   1428
  Lionel Messi                0.8652   0.7812   0.6098   0.7521   0.7639   0.5745    225     63    144   1104
  None                        0.7376   0.2892   0.5263   0.3083   0.7880   0.0

## Per-Class Heatmap

In [6]:
metrics_for_heatmap = ['accuracy', 'precision', 'recall', 'f_beta', 'roc_auc', 'recall_at_fixed_precision']
metric_labels = ['Accuracy', 'Precision', 'Recall', f'F-beta (\u03b2={F1_BETA})', 'ROC-AUC', f'R@P={FIXED_PRECISION_LEVEL}']

class_names_sorted = sorted(pcm.keys())
fig, axes = plt.subplots(1, len(metrics_for_heatmap), figsize=(28, 4))

for col_idx, (metric_key, metric_label) in enumerate(zip(metrics_for_heatmap, metric_labels)):
    ax = axes[col_idx]
    values = [[pcm[cn][metric_key] for cn in class_names_sorted]]
    sns.heatmap(
        values, annot=True, fmt=".3f", cmap="YlOrRd",
        xticklabels=class_names_sorted, yticklabels=[MODEL_DISPLAY],
        ax=ax, vmin=0, vmax=1, cbar=col_idx == len(metrics_for_heatmap) - 1,
    )
    ax.set_title(metric_label, fontsize=12, fontweight='bold')

fig.suptitle(f"Per-Class Metrics Heatmap \u2014 {MODEL_DISPLAY}", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
heatmap_path = EXPERIMENT_DIR / "per_class_metrics_heatmap.png"
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
print(f"\u2713 Saved: {heatmap_path}")
plt.show()

✓ Saved: /app/image_outputs/eval_hog_20260310_205046/per_class_metrics_heatmap.png


## PR and ROC Curves

In [7]:
y_true = detailed["y_true"]
scores = detailed["scores"]
class_names = detailed["class_names"]
colors_palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#8e44ad']

all_precisions, all_recalls = [], []
all_fprs, all_tprs = [], []
class_aucs_pr, class_aucs_roc = [], []

for class_idx, class_name in enumerate(class_names):
    y_true_binary = y_true[:, class_idx]
    if class_name == 'None':
        y_scores = np.array([1 - max(s.values()) if s else 1.0 for s in scores])
    else:
        y_scores = np.array([s.get(class_name, 0.0) for s in scores])

    prec, rec, _ = precision_recall_curve(y_true_binary, y_scores)
    all_precisions.append(prec); all_recalls.append(rec)
    class_aucs_pr.append(sk_auc(rec, prec))

    fpr, tpr, _ = roc_curve(y_true_binary, y_scores)
    all_fprs.append(fpr); all_tprs.append(tpr)
    class_aucs_roc.append(sk_auc(fpr, tpr))

# PR Curve
fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(18, 8))

mean_recall = np.linspace(0, 1, 200)
mean_precision = np.zeros_like(mean_recall)
for p, r in zip(all_precisions, all_recalls):
    mean_precision += np.interp(mean_recall, r[::-1], p[::-1])
mean_precision /= len(all_precisions)
macro_auc_pr = sk_auc(mean_recall, mean_precision)

ax_pr.plot(mean_recall, mean_precision, linewidth=3, color='navy',
           label=f'Macro-avg (AUC={macro_auc_pr:.3f})')
ax_pr.fill_between(mean_recall, mean_precision, alpha=0.15, color='navy')
for ci, (p, r, cn, au) in enumerate(zip(all_precisions, all_recalls, class_names, class_aucs_pr)):
    ax_pr.plot(r, p, linewidth=1.5, alpha=0.5,
               color=colors_palette[ci % len(colors_palette)],
               label=f'{cn} (AUC={au:.3f})')
ax_pr.set_xlabel('Recall', fontsize=11, fontweight='bold')
ax_pr.set_ylabel('Precision', fontsize=11, fontweight='bold')
ax_pr.set_title(f'{MODEL_DISPLAY} \u2014 PR Curve\nMacro AUC: {macro_auc_pr:.3f}', fontsize=13, fontweight='bold')
ax_pr.set_xlim([0, 1]); ax_pr.set_ylim([0, 1.05])
ax_pr.legend(fontsize=9, loc='best')
ax_pr.grid(True, alpha=0.3, linestyle='--')

# ROC Curve
mean_fpr = np.linspace(0, 1, 200)
mean_tpr = np.zeros_like(mean_fpr)
for f, t in zip(all_fprs, all_tprs):
    mean_tpr += np.interp(mean_fpr, f, t)
mean_tpr /= len(all_fprs)
mean_tpr[0] = 0.0
macro_auc_roc = sk_auc(mean_fpr, mean_tpr)

ax_roc.plot(mean_fpr, mean_tpr, linewidth=3, color='navy',
            label=f'Macro-avg (AUC={macro_auc_roc:.3f})')
ax_roc.fill_between(mean_fpr, mean_tpr, alpha=0.15, color='navy')
ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random')
for ci, (f, t, cn, au) in enumerate(zip(all_fprs, all_tprs, class_names, class_aucs_roc)):
    ax_roc.plot(f, t, linewidth=1.5, alpha=0.5,
                color=colors_palette[ci % len(colors_palette)],
                label=f'{cn} (AUC={au:.3f})')
ax_roc.set_xlabel('FPR', fontsize=11, fontweight='bold')
ax_roc.set_ylabel('TPR', fontsize=11, fontweight='bold')
ax_roc.set_title(f'{MODEL_DISPLAY} \u2014 ROC Curve\nMacro AUC: {macro_auc_roc:.3f}', fontsize=13, fontweight='bold')
ax_roc.set_xlim([0, 1]); ax_roc.set_ylim([0, 1.05])
ax_roc.legend(fontsize=9, loc='lower right')
ax_roc.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
curves_path = EXPERIMENT_DIR / "pr_roc_curves.png"
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
print(f"\u2713 Saved: {curves_path}")
plt.show()

✓ Saved: /app/image_outputs/eval_hog_20260310_205046/pr_roc_curves.png


## Summary Report

In [8]:
print("\n" + "=" * 90)
print(f"{MODEL_DISPLAY} EVALUATION \u2014 SUMMARY REPORT")
print("=" * 90)

print(f"\nConfiguration:")
print(f"  Test Set:                        {len(train_items)} images")
print(f"  Face Identification Threshold:   {FACE_IDENTIFICATION_THRESHOLD}")
print(f"  F-beta Parameter:                {F1_BETA}")
print(f"  Fixed Recall Level:              {FIXED_RECALL_LEVEL}")
print(f"  Fixed Precision Level:           {FIXED_PRECISION_LEVEL}")

print(f"\n{'\u2500' * 90}")
print(f"  OVERALL METRICS")
print(f"{'\u2500' * 90}")

metric_rows = [
    ('subset_accuracy', 'Subset Accuracy'),
    ('accuracy_macro', 'Accuracy (macro)'),
    ('f_beta_macro', f'F-beta (macro, \u03b2={F1_BETA})'),
    ('f_beta_micro', f'F-beta (micro, \u03b2={F1_BETA})'),
    ('precision_macro', 'Precision (macro)'),
    ('precision_micro', 'Precision (micro)'),
    ('recall_macro', 'Recall (macro)'),
    ('recall_micro', 'Recall (micro)'),
    ('roc_auc_macro', 'ROC AUC (macro)'),
    ('precision_at_fixed_recall_macro', f'P@R={FIXED_RECALL_LEVEL} (macro)'),
    ('precision_at_fixed_recall_micro', f'P@R={FIXED_RECALL_LEVEL} (micro)'),
    ('recall_at_fixed_precision_macro', f'R@P={FIXED_PRECISION_LEVEL} (macro)'),
    ('recall_at_fixed_precision_micro', f'R@P={FIXED_PRECISION_LEVEL} (micro)'),
    ('identification_rate', 'Identification Rate'),
]

for mkey, mlabel in metric_rows:
    val = result[mkey]
    print(f"  {mlabel:<40} {val:.4f}")

print(f"\n{'\u2500' * 90}")
print(f"  PER-CLASS F-BETA SCORES")
print(f"{'\u2500' * 90}")
for cn in sorted(pcm.keys()):
    print(f"  {cn:<30} {pcm[cn]['f_beta']:.4f}")

print(f"\nAll results saved to: {EXPERIMENT_DIR}")
print("=" * 90)


HOG (face_recognition) EVALUATION — SUMMARY REPORT

Configuration:
  Test Set:                        1536 images
  Face Identification Threshold:   0.3
  F-beta Parameter:                0.4
  Fixed Recall Level:              0.95
  Fixed Precision Level:           0.95

──────────────────────────────────────────────────────────────────────────────────────────
  OVERALL METRICS
──────────────────────────────────────────────────────────────────────────────────────────
  Subset Accuracy                          0.5065
  Accuracy (macro)                         0.8529
  F-beta (macro, β=0.4)                    0.5831
  F-beta (micro, β=0.4)                    0.5715
  Precision (macro)                        0.5805
  Precision (micro)                        0.5625
  Recall (macro)                           0.6322
  Recall (micro)                           0.6348
  ROC AUC (macro)                          nan
  P@R=0.95 (macro)                         0.1698
  P@R=0.95 (micro)           